# 40B · Residual Forcing Test

Notebook 40A showed that under the current setup the **core** and **specialist** models are effectively identical:
- mean \(|r|\) ≈ 0
- var\((r)\) ≈ 0
- corr(core, specialist) ≈ 1

So before asking whether a residual is reusable, we need to **force a real residual to exist**.

This notebook does that by breaking symmetry between the models.

## Core idea

We compare a weaker **core** model against a stronger or differently trained **specialist** model, so that

\[
r(x) = s_{\mathrm{spec}}(x) - s_{\mathrm{core}}(x)
\]

is no longer trivially zero.

## Forced residual families

We test three residual-forcing strategies:

1. **capacity_gap**  
   core = linear logistic model on minimal features  
   specialist = RBF boundary on same data

2. **feature_gap**  
   core = RBF on minimal features  
   specialist = RBF on contextual features

3. **condition_gap**  
   core = RBF trained on mixed train split  
   specialist = RBF trained on condition-specific split

## Outputs

- residual existence panel
- forced residual phase-gap panel
- transfer and orthogonal-gain heatmaps
- residual-only classifier comparison
- forced-residual summary table

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)
        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())
    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

## Features

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_windows_and_features(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    stats = {
        "entropy": np.array([window_entropy(w) for w in windows_n]),
        "unevenness": np.array([unevenness(w) for w in windows_n]),
        "ratio_mean": np.array([ratio_mean(w) for w in windows_n]),
        "symmetry": np.array([reversal_symmetry_score(w) for w in windows_n]),
    }

    if feature_family == "minimal":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["ratio_mean"][i]]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "full":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["symmetry"][i], stats["ratio_mean"][i], ratio_std(windows_n[i])]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(feature_family)

    return windows_n, stats, X

def make_contextual_features(X):
    prev_X = np.vstack([X[0], X[:-1]])
    next_X = np.vstack([X[1:], X[-1]])
    delta_prev = X - prev_X
    delta_next = next_X - X
    curv = next_X - 2 * X + prev_X
    return np.hstack([X, delta_prev, delta_next, curv])

def apply_condition_mask(stats, condition_name):
    if condition_name == "low_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] <= thr
    if condition_name == "high_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] > thr
    if condition_name == "low_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] <= thr
    if condition_name == "high_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] > thr
    raise ValueError(condition_name)

def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std, mean, std

## Models

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    pos = D[D > 0]
    med = np.median(pos) if len(pos) else 1.0
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :]) ** 2, axis=2)
    return np.exp(-gamma * D2)

def fit_linear_boundary(X0_train, X1_train):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]
    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))
    Xtr, _, mean, std = standardize_train_test(X_train, X_train)
    w = fit_logistic_regression(Xtr, y_train, lr=0.12, n_steps=2400, reg=1e-4)
    return {"kind": "linear", "mean": mean, "std": std, "w": w}

def fit_rbf_boundary(X0_train, X1_train, n_proto=20):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]
    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))
    Xtr, _, mean, std = standardize_train_test(X_train, X_train)
    prototypes = choose_prototypes(Xtr, n_proto=min(n_proto, len(Xtr)))
    gamma = estimate_rbf_gamma(Xtr)
    R_train = rbf_features(Xtr, prototypes, gamma)
    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)
    return {"kind": "rbf", "mean": mean, "std": std, "prototypes": prototypes, "gamma": gamma, "w": w}

def score_model(model, X0_test, X1_test):
    m = min(len(X0_test), len(X1_test))
    X0_test = X0_test[:m]
    X1_test = X1_test[:m]
    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))
    Xte = (X_test - model["mean"]) / model["std"]

    if model["kind"] == "linear":
        scores = decision_function_logistic(Xte, model["w"])
    else:
        R_test = rbf_features(Xte, model["prototypes"], model["gamma"])
        scores = decision_function_logistic(R_test, model["w"])

    probs = sigmoid(scores)
    return y_test, scores, probs, X_test

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def evaluate_scores(y_true, scores, probs):
    preds = (probs >= 0.5).astype(int)
    acc = float(np.mean(preds == y_true))
    s0 = scores[y_true == 0]
    s1 = scores[y_true == 1]
    overlap = overlap_coefficient_from_hist(s0, s1, bins=30)
    return {"accuracy": acc, "overlap": overlap}

## Diagnostics

In [ ]:
def fit_linear_regression(X, y, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    I = np.eye(Xb.shape[1]); I[0,0] = 0.0
    beta = np.linalg.solve(Xb.T @ Xb + reg * I, Xb.T @ y)
    return beta

def predict_linear_regression(X, beta):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ beta

def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot <= 1e-12:
        return 0.0
    return float(1.0 - ss_res / ss_tot)

def correlation(x, y):
    x = np.asarray(x); y = np.asarray(y)
    if np.std(x) <= 1e-12 or np.std(y) <= 1e-12:
        return 0.0
    return float(np.corrcoef(x, y)[0,1])

def predict_residual_from_features(X_train, r_train, X_test, use_context=False):
    Xtr = make_contextual_features(X_train) if use_context else X_train
    Xte = make_contextual_features(X_test) if use_context else X_test
    Xtrs, Xtes, _, _ = standardize_train_test(Xtr, Xte)
    beta = fit_linear_regression(Xtrs, r_train, reg=1e-3)
    return predict_linear_regression(Xtes, beta)

def orthogonal_residual(X_train, r_train, X_test, r_test, use_context=False):
    r_pred = predict_residual_from_features(X_train, r_train, X_test, use_context=use_context)
    return r_pred, r_test - r_pred

def residual_only_classifier(r_train, y_train, r_test, y_test):
    Xtr = r_train.reshape(-1, 1)
    Xte = r_test.reshape(-1, 1)
    Xtrs, Xtes, _, _ = standardize_train_test(Xtr, Xte)
    w = fit_logistic_regression(Xtrs, y_train, lr=0.1, n_steps=1600, reg=1e-4)
    scores = decision_function_logistic(Xtes, w)
    probs = predict_proba_logistic(Xtes, w)
    return evaluate_scores(y_test, scores, probs)

def evaluate_orthogonal_gain(s_core, y_true, r_orth, alpha_grid=None):
    if alpha_grid is None:
        alpha_grid = np.linspace(0.0, 1.5, 16)
    base_eval = evaluate_scores(y_true, s_core, sigmoid(s_core))
    best_gain, best_alpha = -np.inf, 0.0
    for a in alpha_grid:
        s_new = s_core + a * r_orth
        cur = evaluate_scores(y_true, s_new, sigmoid(s_new))
        gain = cur["overlap"] - base_eval["overlap"]
        if gain > best_gain:
            best_gain, best_alpha = gain, float(a)
    return best_alpha, float(best_gain)

def residual_existence_metrics(residual_raw, s_core, s_spec, eps=1e-8):
    return {
        "residual_mean_abs_raw": float(np.mean(np.abs(residual_raw))),
        "residual_variance_raw": float(np.var(residual_raw)),
        "near_zero_frac": float(np.mean(np.abs(residual_raw) < eps)),
        "score_corr": correlation(s_core, s_spec),
    }

## Forced residual extractor

In [ ]:
def extract_forced_residual(X0_low, X1_low, X0_high, X1_high, eval_mode, forcing_mode):
    train0 = np.vstack([X0_low, X0_high])
    train1 = np.vstack([X1_low, X1_high])

    if eval_mode == "low":
        X0_test, X1_test = X0_low, X1_low
        X0_cross, X1_cross = X0_high, X1_high
    elif eval_mode == "high":
        X0_test, X1_test = X0_high, X1_high
        X0_cross, X1_cross = X0_low, X1_low
    else:
        X0_test, X1_test = train0, train1
        X0_cross, X1_cross = train0, train1

    if forcing_mode == "capacity_gap":
        core_model = fit_linear_boundary(train0, train1)
        spec_model = fit_rbf_boundary(train0, train1, n_proto=20)

        y_true, s_core, p_core, X_test = score_model(core_model, X0_test, X1_test)
        _, s_spec, _, _ = score_model(spec_model, X0_test, X1_test)

        y_cross, s_core_cross, _, X_cross = score_model(core_model, X0_cross, X1_cross)
        _, s_spec_cross, _, _ = score_model(spec_model, X0_cross, X1_cross)

    elif forcing_mode == "feature_gap":
        ctx_train0, ctx_train1 = make_contextual_features(train0), make_contextual_features(train1)
        ctx_test0, ctx_test1 = make_contextual_features(X0_test), make_contextual_features(X1_test)
        ctx_cross0, ctx_cross1 = make_contextual_features(X0_cross), make_contextual_features(X1_cross)

        core_model = fit_rbf_boundary(train0, train1, n_proto=20)
        spec_model = fit_rbf_boundary(ctx_train0, ctx_train1, n_proto=20)

        y_true, s_core, p_core, X_test = score_model(core_model, X0_test, X1_test)
        _, s_spec, _, _ = score_model(spec_model, ctx_test0, ctx_test1)

        y_cross, s_core_cross, _, X_cross = score_model(core_model, X0_cross, X1_cross)
        _, s_spec_cross, _, _ = score_model(spec_model, ctx_cross0, ctx_cross1)

    elif forcing_mode == "condition_gap":
        core_model = fit_rbf_boundary(train0, train1, n_proto=20)

        if eval_mode == "low":
            spec_model = fit_rbf_boundary(X0_low, X1_low, n_proto=20)
        elif eval_mode == "high":
            spec_model = fit_rbf_boundary(X0_high, X1_high, n_proto=20)
        else:
            spec_model = fit_rbf_boundary(X0_high, X1_high, n_proto=20)

        y_true, s_core, p_core, X_test = score_model(core_model, X0_test, X1_test)
        _, s_spec, _, _ = score_model(spec_model, X0_test, X1_test)

        y_cross, s_core_cross, _, X_cross = score_model(core_model, X0_cross, X1_cross)
        _, s_spec_cross, _, _ = score_model(spec_model, X0_cross, X1_cross)

    else:
        raise ValueError(forcing_mode)

    residual_raw = s_spec - s_core
    residual = residual_raw / (np.std(residual_raw) + 1e-6)

    residual_raw_cross = s_spec_cross - s_core_cross
    residual_cross = residual_raw_cross / (np.std(residual_raw_cross) + 1e-6)

    return {
        "y_true": y_true,
        "X_test": X_test,
        "s_core": s_core,
        "s_spec": s_spec,
        "p_core": p_core,
        "residual_raw": residual_raw,
        "residual": residual,
        "core_uncertainty": p_core * (1.0 - p_core),
        "core_margin": np.abs(s_core),
        "core_error": ((p_core >= 0.5).astype(int) != y_true).astype(float),
        "y_cross": y_cross,
        "X_cross": X_cross,
        "residual_cross": residual_cross,
    }

## Experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 100
height_block = (0, 400)
alpha_grid = np.linspace(0.0, 1.5, 16)

systems = {
    "entropy": ("low_entropy", "high_entropy"),
    "unevenness": ("low_unevenness", "high_unevenness"),
}
tasks = {
    "zeta_vs_GUE": ("zeta", "gue"),
    "zeta_vs_Poisson": ("zeta", "poisson"),
}
forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]

## Base gap slices

In [ ]:
start, stop = height_block
base_gaps = {
    "zeta": zeta_gaps_full[start:stop],
    "poisson": poisson_gaps_full[start:stop],
    "gue": gue_gaps_full[:max(stop - start + 300, 900)],
}

## Build condition-specific feature sets

In [ ]:
def get_conditioned_features(gaps, condition_name, k=5, feature_family="minimal", n=100):
    _, stats, X = build_windows_and_features(gaps, feature_family=feature_family, k=k)
    mask = apply_condition_mask(stats, condition_name)
    Xc = X[mask]
    return Xc[:min(len(Xc), n)]

conditioned = {}
for k in window_sizes:
    conditioned[k] = {}
    for cond_lo, cond_hi in systems.values():
        for cond in [cond_lo, cond_hi]:
            for ensemble_name, gaps in base_gaps.items():
                conditioned[k][(ensemble_name, cond)] = get_conditioned_features(
                    gaps, cond, k=k, feature_family=feature_family, n=sample_size
                )

## Main forced-residual sweep

In [ ]:
results = []

for system_name, (cond_lo, cond_hi) in systems.items():
    for task_name, (ens0, ens1) in tasks.items():
        for k in window_sizes:
            X0_low = conditioned[k][(ens0, cond_lo)]
            X1_low = conditioned[k][(ens1, cond_lo)]
            X0_high = conditioned[k][(ens0, cond_hi)]
            X1_high = conditioned[k][(ens1, cond_hi)]

            m = min(len(X0_low), len(X1_low), len(X0_high), len(X1_high), sample_size)
            if m < 40:
                continue

            X0_low = X0_low[:m]; X1_low = X1_low[:m]
            X0_high = X0_high[:m]; X1_high = X1_high[:m]

            for forcing_mode in forcing_modes:
                for eval_mode in ["low", "high", "mixed"]:
                    data = extract_forced_residual(X0_low, X1_low, X0_high, X1_high, eval_mode, forcing_mode)
                    existence = residual_existence_metrics(data["residual_raw"], data["s_core"], data["s_spec"])

                    residual = data["residual"]
                    X_test = data["X_test"]

                    r_pred_basic = predict_residual_from_features(X_test, residual, X_test, use_context=False)
                    r_pred_ctx = predict_residual_from_features(X_test, residual, X_test, use_context=True)

                    transfer_pred = predict_residual_from_features(
                        X_test, residual, data["X_cross"], use_context=True
                    )
                    transfer_r2 = r2_score(data["residual_cross"], transfer_pred)

                    _, r_orth = orthogonal_residual(X_test, residual, X_test, residual, use_context=True)
                    best_alpha, orth_gain = evaluate_orthogonal_gain(data["s_core"], data["y_true"], r_orth, alpha_grid=alpha_grid)

                    resid_only = residual_only_classifier(residual, data["y_true"], residual, data["y_true"])

                    results.append({
                        "system": system_name,
                        "task": task_name,
                        "k": k,
                        "forcing_mode": forcing_mode,
                        "eval": eval_mode,
                        **existence,
                        "r2_basic": r2_score(residual, r_pred_basic),
                        "r2_ctx": r2_score(residual, r_pred_ctx),
                        "corr_uncertainty": correlation(np.abs(residual), data["core_uncertainty"]),
                        "corr_error": correlation(np.abs(residual), data["core_error"]),
                        "corr_margin_proxy": correlation(np.abs(residual), 1.0 / (1.0 + data["core_margin"])),
                        "transfer_r2": transfer_r2,
                        "orth_best_alpha": best_alpha,
                        "orth_gain": orth_gain,
                        "residual_only_overlap": resid_only["overlap"],
                        "residual_only_accuracy": resid_only["accuracy"],
                    })

len(results)

## Plot 0 · Residual existence panel

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True)

for ax, metric, title in zip(
    axes,
    ["residual_mean_abs_raw", "residual_variance_raw", "score_corr"],
    ["mean |residual_raw|", "var(residual_raw)", "corr(core, specialist)"]
):
    for forcing_mode in forcing_modes:
        rows = [r for r in results if r["forcing_mode"] == forcing_mode and r["system"] == "entropy" and r["task"] == "zeta_vs_GUE" and r["eval"] == "mixed"]
        rows = sorted(rows, key=lambda x: x["k"])
        ax.plot([r["k"] for r in rows], [r[metric] for r in rows], marker="o", label=forcing_mode)
    ax.set_title(title)
    ax.set_xlabel("window size k")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Plot 1 · Forced residual phase-gap panel

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in results if r["system"] == system_name and r["task"] == "zeta_vs_GUE" and r["eval"] == "mixed"]
    x = np.arange(len(window_sizes))
    width = 0.22
    for offset, forcing_mode in zip([-width, 0, width], forcing_modes):
        sub = sorted([r for r in rows if r["forcing_mode"] == forcing_mode], key=lambda x: x["k"])
        vals = [r["residual_mean_abs_raw"] for r in sub]
        ax.bar(x + offset, vals, width, label=forcing_mode)
    ax.set_xticks(x, window_sizes)
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("mean |residual_raw|")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Plot 2 · Transfer heatmaps

In [ ]:
def build_matrix(metric, system_name, task_name, forcing_mode):
    rows = [r for r in results if r["system"] == system_name and r["task"] == task_name and r["forcing_mode"] == forcing_mode]
    return np.array([
        [next(r for r in rows if r["k"] == k and r["eval"] == "low")[metric] for k in window_sizes],
        [next(r for r in rows if r["k"] == k and r["eval"] == "high")[metric] for k in window_sizes],
        [next(r for r in rows if r["k"] == k and r["eval"] == "mixed")[metric] for k in window_sizes],
    ])

fig, axes = plt.subplots(3, 2, figsize=(10, 11), sharex=True, sharey=True)
for i, forcing_mode in enumerate(forcing_modes):
    for j, system_name in enumerate(["entropy", "unevenness"]):
        ax = axes[i, j]
        M = build_matrix("transfer_r2", system_name, "zeta_vs_GUE", forcing_mode)
        im = ax.imshow(M, aspect="auto", origin="upper")
        ax.set_title(f"{forcing_mode} · {system_name}")
        ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
        ax.set_yticks(np.arange(3), ["low eval", "high eval", "mixed eval"])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="cross-condition transfer $R^2$")
plt.tight_layout()
plt.show()

## Plot 3 · Orthogonal-gain heatmaps

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 11), sharex=True, sharey=True)
for i, forcing_mode in enumerate(forcing_modes):
    for j, system_name in enumerate(["entropy", "unevenness"]):
        ax = axes[i, j]
        M = build_matrix("orth_gain", system_name, "zeta_vs_GUE", forcing_mode)
        im = ax.imshow(M, aspect="auto", origin="upper")
        ax.set_title(f"{forcing_mode} · {system_name}")
        ax.set_xticks(np.arange(len(window_sizes)), window_sizes)
        ax.set_yticks(np.arange(3), ["low eval", "high eval", "mixed eval"])
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="best orthogonal residual gain")
plt.tight_layout()
plt.show()

## Plot 4 · Residual-only classifier comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    rows = [r for r in results if r["system"] == system_name and r["task"] == "zeta_vs_GUE" and r["eval"] == "mixed"]
    rows = sorted(rows, key=lambda x: x["forcing_mode"] + str(x["k"]))
    labels = [f'{r["forcing_mode"][:4]}-{r["k"]}' for r in rows]
    vals = [r["residual_only_overlap"] for r in rows]
    ax.bar(np.arange(len(vals)), vals)
    ax.set_xticks(np.arange(len(vals)), labels, rotation=45)
    ax.set_title(system_name)
    ax.set_ylabel("residual-only overlap")

plt.tight_layout()
plt.show()

## Forced-residual summary table

In [ ]:
summary_rows = []
for forcing_mode in forcing_modes:
    for system_name in ["entropy", "unevenness"]:
        for task_name in ["zeta_vs_GUE", "zeta_vs_Poisson"]:
            for k in window_sizes:
                row = next(r for r in results if r["forcing_mode"] == forcing_mode and r["system"] == system_name and r["task"] == task_name and r["eval"] == "mixed" and r["k"] == k)
                summary_rows.append({
                    "forcing_mode": forcing_mode,
                    "system": system_name,
                    "task": task_name,
                    "k": k,
                    "residual_mean_abs_raw": row["residual_mean_abs_raw"],
                    "residual_variance_raw": row["residual_variance_raw"],
                    "score_corr": row["score_corr"],
                    "r2_ctx": row["r2_ctx"],
                    "transfer_r2": row["transfer_r2"],
                    "orth_gain": row["orth_gain"],
                    "residual_only_overlap": row["residual_only_overlap"],
                })
summary_rows

## Reading guide

- **capacity_gap** asks whether a stronger nonlinear specialist creates a meaningful residual over a weaker core.
- **feature_gap** asks whether contextual features create a residual beyond the minimal feature space.
- **condition_gap** asks whether condition-specific training creates a residual over a mixed core.

A useful forced residual should show:
- nonzero mean |residual|
- nontrivial residual variance
- score correlation below perfect collapse
- transfer or orthogonal gain above zero
- residual-only overlap below 1 and preferably task-sensitive